# 01 — Data Audit

**PrioMed-Rec | Sub-Task 2**

Objective: Load all Synthea CSV files, validate schema, compute DQI metrics, and persist `data/interim/dqi_report.json`.

Run from the repo root: `jupyter notebook notebooks/01_data_audit.ipynb`

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import json
from pathlib import Path

BASE_DIR = '../data/raw/synthea'
print('Working from:', Path(BASE_DIR).resolve())

## 1. Load all tables and inspect shape

In [ ]:
from src.data.loaders import (
    load_patients, load_encounters, load_medications,
    load_medication_requests, load_medication_administrations,
    load_conditions, load_allergies,
    load_primedrec_master, load_primedrec_risk_features,
    load_validation_report
)

patients    = load_patients(BASE_DIR)
encounters  = load_encounters(BASE_DIR)
medications = load_medications(BASE_DIR)
med_req     = load_medication_requests(BASE_DIR)
med_admin   = load_medication_administrations(BASE_DIR)
conditions  = load_conditions(BASE_DIR)
allergies   = load_allergies(BASE_DIR)
master      = load_primedrec_master(BASE_DIR)
risk_feat   = load_primedrec_risk_features(BASE_DIR)
val_report  = load_validation_report(BASE_DIR)

tables = {
    'patients': patients, 'encounters': encounters,
    'medications': medications, 'medication_requests': med_req,
    'medication_administrations': med_admin, 'conditions': conditions,
    'allergies': allergies, 'primedrec_master': master,
    'primedrec_risk_features': risk_feat
}

for name, df in tables.items():
    print(f'{name}: {df.shape}')

## 2. Validation report

In [ ]:
val_report

## 3. Schema & dtypes per table

In [ ]:
for name, df in tables.items():
    print(f'\n=== {name} ===')
    print(df.dtypes.to_string())

## 4. Null rate per column

In [ ]:
for name, df in tables.items():
    null_rates = (df.isnull().sum() / len(df) * 100).round(2)
    high_null = null_rates[null_rates > 0]
    if not high_null.empty:
        print(f'\n=== {name} — columns with nulls ===')
        print(high_null.to_string())
    else:
        print(f'\n=== {name}: no nulls ===')

## 5. Sample values per table

In [ ]:
patients.head(3)

In [ ]:
encounters.head(3)

In [ ]:
med_req.head(3)

In [ ]:
med_admin.head(3)

In [ ]:
master.head(3)

## 6. Encounter class distribution

In [ ]:
print('encounter_class value counts:')
print(encounters['encounter_class'].value_counts())

## 7. Referential integrity spot-checks

In [ ]:
from src.data.validator import referential_integrity_check

checks = [
    (encounters, patients, 'patient_id', 'patient_id', 'encounters', 'patients'),
    (med_req, encounters, 'encounter_id', 'encounter_id', 'medication_requests', 'encounters'),
    (allergies, patients, 'patient_id', 'patient_id', 'allergies', 'patients'),
]

for args in checks:
    result = referential_integrity_check(*args)
    status = 'PASS' if result['passed'] else 'FAIL'
    print(f'[{status}] {result["check"]} | orphans: {result["orphan_count"]} ({result["orphan_rate"]*100:.1f}%)')

## 8. Duplicate column check (primedrec_master)

In [ ]:
from src.data.validator import duplicate_column_check

dup = duplicate_column_check(master, 'primedrec_master')
for pair in dup['duplicate_pairs']:
    same = master[pair['col_x']].equals(master[pair['col_y']])
    print(f"  {pair['col_x']} == {pair['col_y']}: {same}")

## 9. Run full DQI audit and save report

In [ ]:
from src.data.validator import run_full_audit, save_dqi_report

report = run_full_audit(BASE_DIR)
save_dqi_report(report, '../data/interim/dqi_report.json')
print('Audit complete.')

## Summary of DQI Findings

| Issue ID | Table | Finding | Decision |
|---|---|---|---|
| DQI-001 | allergies | `patient_id` 100% null | Exclude allergy features — see DECISION_LOG.md |
| DQI-002 | primedrec_master | Duplicate columns `_x/_y` | Use `_x` variants |
| DQI-003 | encounters | `encounter_class_display` 100% null | Use `encounter_class` |
| DQI-004 | medication_requests | 36.1% null medication names | Prefer medication_administrations |
| DQI-005 | medication_requests | dose/route/freq 100% null | Mark dose/route/freq rules as UNRESOLVED |
| DQI-006 | medication_requests | 50.9% orphan encounter_id | Use only matched encounters for cohort |

See `docs/DECISION_LOG.md` for full rationale.